<a href="https://colab.research.google.com/github/Rachani02/Statistical-Learning-e22282/blob/main/Assignment_7b_Gaussian_Mixture_Model_Clustering_as_Conditional_Updating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Gaussian Mixture Clustering as Conditional Updating


1. Deriving the Marginal Density

By the law of total probability for a continuous random variable $X_i$ mixed over a discrete latent space $C_i$, we sum the joint distribution over all possible realizations of $C_i$:

$$p(x_i) = \sum_{k=1}^K p(x_i, C_i = k) = \sum_{k=1}^K P(C_i = k) p(x_i \mid C_i = k)$$

Substituting the prior $P(C_i = k) = \phi_k$ and the conditional Gaussian density $p(x_i \mid C_i = k) = \mathscr N(x_i \mid \mu_k, \Sigma_k)$, we obtain:

$$p(x_i) = \sum_{k=1}^K \phi_k \mathscr N(x_i \mid \mu_k, \Sigma_k)$$

This is called a Gaussian mixture density because the final density profile is formed by "mixing" or linearly combining $K$ individual multivariate Gaussian base components, where the mixing proportions are controlled by the prior probabilities $\phi_k$.

2. Deriving the Posterior Cluster Probability

By Bayes' rule, the posterior probability of a discrete label given a continuous observation is:

$$P(C_i = k \mid X_i = x_i) = \frac{p(X_i = x_i \mid C_i = k) P(C_i = k)}{p(x_i)}$$

Expanding the marginal density denominator $p(x_i)$ using the result from Part 1 yields:

$$P(C_i = k \mid X_i = x_i) = \frac{P(X_i = x_i \mid C_i = k) P(C_i = k)}{\sum_{j=1}^K P(X_i = x_i \mid C_i = j) P(C_i = j)}$$

Substituting the model specific choices ($P(C_i=k)=\phi_k$ and $p(x_i \mid C_i=k) = \mathscr N(x_i \mid \mu_k, \Sigma_k)$):

$$\gamma_{ik} = P(C_i = k \mid X_i = x_i) = \frac{\phi_k \mathscr N(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathscr N(x_i \mid \mu_j, \Sigma_j)}$$

$\gamma_{ik}$ is a valid posterior probability because it is derived by updating our prior belief $\phi_k$ using the relative evidence provided by the generative component likelihood $\mathscr N(x_i \mid \mu_k, \Sigma_k)$, naturally satisfying $\gamma_{ik} \ge 0$ and $\sum_{k=1}^K \gamma_{ik} = 1$.

3. One-Hot Encoding of the Latent Cluster Variable

Since $Z_{ik}$ is a binary indicator random variable taking values in $\{0, 1\}$, its conditional expectation is exactly the probability of it evaluating to 1:

$$\mathbb E[Z_{ik} \mid X_i = x_i] = 1 \cdot P(Z_{ik} = 1 \mid X_i = x_i) + 0 \cdot P(Z_{ik} = 0 \mid X_i = x_i)$$

$$\mathbb E[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i) = \gamma_{ik}$$

Extending this element-wise to the vector $Z_i$:

$$\mathbb E[Z_i \mid X_i = x_i] = \begin{bmatrix} \mathbb E[Z_{i1} \mid X_i = x_i] \\ \vdots \\ \mathbb E[Z_{iK} \mid X_i = x_i] \end{bmatrix} = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$

Conclusion: The soft cluster assignment vector in a GMM is mathematically identical to the conditional expectation $\mathbb E[Z_i \mid X_i = x_i]$ of the latent indicator state.

4. From Soft Assignment to Hard Clustering

Soft Clustering: Keeps the full allocation vector $\mathbb E[Z_i \mid X_i = x_i]$, representing cluster ambiguity continuously (e.g., a data point can belong $60\%$ to Cluster 1 and $40\%$ to Cluster 2).

Hard Clustering: Projects this vector onto the vertices of the standard simplex by taking a deterministic maximum operator ($\widehat C_i = \operatorname{arg\,max}_k \gamma_{ik}$), discarding information about relative uncertainty.

5. Conditional Expectation of the Observation Given the Cluster

Given that $X_i \mid C_i = k \sim \mathscr N(\mu_k, \Sigma_k)$, the conditional expectation is simply the mean parameter of that specific distribution:

$$\mathbb E[X_i \mid C_i = k] = \int_{\mathbb R^d} x_i \mathscr N(x_i \mid \mu_k, \Sigma_k) dx_i = \mu_k$$

Comparison:

$\mathbb E[Z_i \mid X_i = x_i]$ maps from feature space to latent space, determining the distribution over group categories for a specific observed coordinate vector.

$\mathbb E[X_i \mid C_i = k]$ maps from latent space to feature space, defining the geometric coordinate expected from an unobserved prototypical point belonging strictly to category $k$.

6. The Complete-Data Likelihood

Given the complete data log-likelihood formula:

$$p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr N(x_i\mid \mu_k,\Sigma_k) \right]^{z_{ik}}$$

Applying the natural logarithm transforms the product operators into summations:

$$\ell_c = \log \left( \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr N(x_i\mid \mu_k,\Sigma_k) \right]^{z_{ik}} \right) = \sum_{i=1}^n \sum_{k=1}^K \log \left( \left[ \phi_k \mathscr N(x_i\mid \mu_k,\Sigma_k) \right]^{z_{ik}} \right)$$

Using the power rule for logarithms ($\log(a^b) = b\log a$):

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathscr N(x_i\mid \mu_k,\Sigma_k) \right]$$

Interpretation: If $z_{ik}$ were known, this optimization problem decouples completely across clusters because $z_{ik}$ acts as a hard filter. The optimal $\mu_k$ and $\Sigma_k$ could be calculated in closed form using standard independent Gaussian Maximum Likelihood Estimation (MLE) calculations restricted solely to the points belonging to group $k$.

7. The EM Interpretation

Substituting $z_{ik} \rightsquigarrow \gamma_{ik}$ directly yields the objective function $Q$:

$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \log \phi_k + \log \mathscr N(x_i\mid \mu_k,\Sigma_k) \right]$$

The Expectation step is fundamentally a conditional update. Rather than guessing labels blindly, it leverages the current system parameters to recalculate the optimal probability measure over the hidden components conditional on the fixed properties of the empirical observations.

8. Parameter Updates

To optimize $Q$ with respect to $\phi_k$ under the constraint $\sum_{k=1}^K \phi_k = 1$, construct the Lagrangian:

$$\mathcal L(\phi, \lambda) = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \log \phi_k + \lambda \left( 1 - \sum_{k=1}^K \phi_k \right)$$

Setting $\frac{\partial \mathcal L}{\partial \phi_k} = 0 \implies \frac{\sum_{i=1}^n \gamma_{ik}}{\phi_k} - \lambda = 0 \implies \phi_k = \frac{N_k}{\lambda}$. Summing over $k$ proves $\lambda = n$, hence $\phi_k^{\text{new}} = \frac{N_k}{n}$.For $\mu_k$, expanding the multivariate Gaussian term inside the summation gives:

$$\frac{\partial Q}{\partial \mu_k} = \sum_{i=1}^n \gamma_{ik} \Sigma_k^{-1} (x_i - \mu_k) = 0 \implies \Sigma_k^{-1} \sum_{i=1}^n \gamma_{ik} x_i = \Sigma_k^{-1} \left( \sum_{i=1}^n \gamma_{ik} \right) \mu_k$$

Multiplying by $\Sigma_k$ and isolating $\mu_k$ yields:

$$\mu_k^{\text{new}} = \frac{\sum_{i=1}^n \gamma_{ik} x_i}{\sum_{i=1}^n \gamma_{ik}} = \frac{1}{N_k}\sum_{i=1}^n \gamma_{ik} x_i$$

Taking the matrix derivative of $Q$ with respect to $\Sigma_k^{-1}$ yields standard covariance optimization forms, resulting in:

$$\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} (x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T$$

$\gamma_{ik}$ scales how heavily $x_i$ impacts cluster $k$'s geometry. If a point has $\gamma_{ik} = 0.2$, it contributes exactly $20\%$ of its dimensional properties toward the calculations of that component's updated mean and spatial variance structure.

9. Summary Interpretation

Gaussian Mixture Model (GMM) clustering can be comprehensively formalized as an alternating process of statistical conditional updating. The parameters $\phi_k$ initialize the process as the prior probabilities of group membership. Once data is observed, the spatial alignment of each feature coordinate $x_i$ is evaluated against each component cluster via the generative likelihood density $\mathscr N(x_i \mid \mu_k, \Sigma_k)$. Through Bayes' theorem, these prior weights and conditional likelihoods are reconciled to construct the responsibility metric $\gamma_{ik}$, which represents the posterior probability of cluster identity.

Geometrically, this produces the continuous soft assignment vector $\mathbb E[Z_i \mid X_i = x_i]$. In the subsequent Maximization step, these posterior cluster membership profiles serve as fractional weights to shift the mean centers and alter the shape of the covariance boundaries. Ultimately, Gaussian mixture clustering avoids arbitrary hard splits, operating as a probabilistic framework built directly upon the conditional expectations of latent variables.

In [2]:
import os
import kagglehub
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
import plotly.graph_objects as go
from plotly.subplots import make_subplots

class GMMFinancialSegmenter:
    def __init__(self, data_dir_path: str):
        """
        Initializes the segmenter by locating the dataset file in the kagglehub
        directory and preprocessing the CC DATA.
        """
        # Locate the CSV file inside the downloaded directory path
        target_file = "CC GENERAL.csv"
        csv_path = os.path.join(data_dir_path, target_file)

        # Fallback if the files inside the kagglehub output directory are nested differently
        if not os.path.exists(csv_path):
            for root, dirs, files in os.walk(data_dir_path):
                if target_file in files:
                    csv_path = os.path.join(root, target_file)
                    break

        print(f"Loading dataset from resolved local path: {csv_path}")
        df = pd.read_csv(csv_path)

        # Isolate the requested features and drop missing entries to keep validation pure
        self.features = ['PURCHASES', 'CREDIT_LIMIT']
        df_clean = df[self.features].dropna()

        self.X_raw = df_clean.values

        # Scale features to equalize variance impacts across both axes
        self.scaler = StandardScaler()
        self.X_scaled = self.scaler.fit_transform(self.X_raw)

        # 80/20 out-of-sample data splitting
        self.X_train, self.X_test = train_test_split(self.X_scaled, test_size=0.2, random_state=42)
        self.gmm = None

    def fit(self, n_components: int = 3):
        """Fits the GMM structure using the EM algorithm on training parameters."""
        print(f"Executing EM Algorithm for K={n_components} components...")
        self.gmm = GaussianMixture(n_components=n_components, random_state=42, init_params='kmeans')
        self.gmm.fit(self.X_train)

        print("\n--- Convergence Summary ---")
        print(f"EM Successfully Converged: {self.gmm.converged_}")
        print(f"Iterations Required: {self.gmm.n_iter_}")

        # Compute generalizability metric via unseen out-of-sample likelihood metrics
        test_ll = self.gmm.score(self.X_test)
        print(f"Average Out-of-Sample Log-Likelihood: {test_ll:.4f}\n")

    def plot_empirical_density(self):
        """
        Figure 1: Robust 2D Density Heatmap with marginal distribution histograms.
        Avoids the Plotly Express continuous color bug by building the subplots manually using graph_objects.
        """
        x_data = self.X_train[:, 0]
        y_data = self.X_train[:, 1]

        # Initialize a 2x2 subplot layout with shared axes (top-right cell is empty)
        fig = make_subplots(
            rows=2, cols=2,
            row_heights=[0.2, 0.8],
            column_widths=[0.8, 0.2],
            shared_xaxes=True,
            shared_yaxes=True,
            horizontal_spacing=0.03,
            vertical_spacing=0.03
        )

        # 1. Main 2D Histogram (Bottom-Left)
        fig.add_trace(
            go.Histogram2d(
                x=x_data,
                y=y_data,
                colorscale="Viridis",
                colorbar=dict(title="Density Count", x=1.1)
            ),
            row=2, col=1
        )

        # 2. X Marginal Distribution (Top-Left)
        fig.add_trace(
            go.Histogram(
                x=x_data,
                marker=dict(color="rgb(49, 130, 189)"),  # Safe discrete color
                showlegend=False
            ),
            row=1, col=1
        )

        # 3. Y Marginal Distribution (Bottom-Right)
        fig.add_trace(
            go.Histogram(
                y=y_data,
                marker=dict(color="rgb(49, 130, 189)"),  # Safe discrete color
                showlegend=False
            ),
            row=2, col=2
        )

        # Apply axes labels and clean style settings
        fig.update_layout(
            title="Figure 1: Empirical 2D Density Heatmap of Training Data (Standardized Scale)",
            template="plotly_white",
            xaxis_title=f"{self.features[0]} (Standardized)" if hasattr(fig, 'xaxis2') else None,
            yaxis_title=f"{self.features[1]} (Standardized)" if hasattr(fig, 'yaxis') else None,
            width=850,
            height=750
        )

        # Explicitly tag the correct subplot axes labels
        fig.update_xaxes(title_text=f"{self.features[0]} (Standardized)", row=2, col=1)
        fig.update_yaxes(title_text=f"{self.features[1]} (Standardized)", row=2, col=1)

        fig.show()

    def _generate_contour_grid(self):
        """Helper to compute maximum posterior responsibilities over an evaluation mesh coordinate grid."""
        x_min, x_max = self.X_scaled[:, 0].min() - 0.5, self.X_scaled[:, 0].max() + 0.5
        y_min, y_max = self.X_scaled[:, 1].min() - 0.5, self.X_scaled[:, 1].max() + 0.5

        x_grid = np.linspace(x_min, x_max, 200)
        y_grid = np.linspace(y_min, y_max, 200)
        xx, yy = np.meshgrid(x_grid, y_grid)
        grid_points = np.c_[xx.ravel(), yy.ravel()]

        # Calculate full matrix of gammas: shapes (N_grid, K)
        gamma_matrix = self.gmm.predict_proba(grid_points)
        # Isolate the max posterior value to illustrate boundaries and transition states
        max_gamma = np.max(gamma_matrix, axis=1).reshape(xx.shape)

        return x_grid, y_grid, max_gamma

    def plot_assignments(self, dataset_type: str = "train"):
        """Figures 2 & 3: Plot data points on top of continuous maximum posterior contour spaces."""
        x_grid, y_grid, max_gamma = self._generate_contour_grid()
        data_to_plot = self.X_train if dataset_type == "train" else self.X_test

        # Predict hard clustering classes for discrete scatter visualization overlay
        labels = self.gmm.predict(data_to_plot)

        fig = go.Figure()

        # Add continuous surface of maximum posterior expectations
        fig.add_trace(go.Contour(
            x=x_grid, y=y_grid, z=max_gamma,
            colorscale="Electric",
            contours_coloring="heatmap",
            line_width=0,
            opacity=0.85,
            name="Max Responsibility",
            colorbar=dict(title="Max $\\gamma_{ik}$")
        ))

        # Scatter actual data coordinates classified colored by hard metrics
        for k in range(self.gmm.n_components):
            cluster_mask = (labels == k)
            fig.add_trace(go.Scatter(
                x=data_to_plot[cluster_mask, 0],
                y=data_to_plot[cluster_mask, 1],
                mode='markers',
                marker=dict(size=5, line=dict(width=0.5, color='white')),
                name=f'Hard Cluster Assignment {k+1}'
            ))

        fig.update_layout(
            title=f"Figure {2 if dataset_type=='train' else 3}: Maximum Posterior Responsibility Overlaid with {dataset_type.capitalize()} Sets",
            xaxis_title=f"{self.features[0]} (Standardized)",
            yaxis_title=f"{self.features[1]} (Standardized)",
            template="plotly_white"
        )
        fig.show()

# --- Execution Block ---
# Download the latest version of the CC General dataset via kagglehub
path = kagglehub.dataset_download("arjunbhasin2013/ccdata")
print("Path to dataset files:", path)

# Pass the resolved download directory to the pipeline
segmenter = GMMFinancialSegmenter(path)
segmenter.fit(n_components=3)

# Display the validation plots sequentially
segmenter.plot_empirical_density()
segmenter.plot_assignments("train")
segmenter.plot_assignments("test")

Using Colab cache for faster access to the 'ccdata' dataset.
Path to dataset files: /kaggle/input/ccdata
Loading dataset from resolved local path: /kaggle/input/ccdata/CC GENERAL.csv
Executing EM Algorithm for K=3 components...

--- Convergence Summary ---
EM Successfully Converged: True
Iterations Required: 19
Average Out-of-Sample Log-Likelihood: -1.6465



10. Evaluation of the Resulting Plots

Figure 1 (Empirical 2D Density):

The raw data reveals a heavy density concentration near the origin (low purchases, low credit limits) with structural "arms" extending outward (high credit limit with low purchases, and a distinct diagonal representing people whose purchases scale with credit limit). The marginal distributions clearly illustrate that a standard unimodal distribution (like a single Gaussian) would fail to capture this complex shape, thus justifying the choice of a multi-component Mixture Model ($K=3$).

Figures 2 & 3 (Assignment Plots):

The EM algorithm successfully carves the feature space into three regions. However, unlike algorithms like K-Means which draw sharp linear boundaries, the GMM produces soft, curved boundaries. Observing Figure 3 (the Test set) reveals that several out-of-sample data points sit directly on top of the transition zones, illustrating that forcing these points into a single hard-assigned cluster discards significant physical ambiguity.

How the Contour Map Visualizes the Soft Assignment Expectation $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$

In Part 3, you proved analytically that the soft cluster assignment vector is exactly the conditional expectation of the latent indicator variable:

$$\mathbb{E}[Z_i \mid X_i = x_i] = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$

where each element $\gamma_{ik} = P(C_i = k \mid X_i = x_i)$ represents the posterior probability (responsibility) that data point $x_i$ belongs to cluster $k$.The background contour map visually demonstrates this vector across a continuous coordinate grid $x_{\text{grid}}$ by plotting the magnitude of the maximum component of this expectation vector:

$$f(x_{\text{grid}}) = \max_{1 \le k \le K} \mathbb{E}[Z_{ik} \mid X_i = x_{\text{grid}}] = \max_k \gamma_{\text{grid}, k}$$

This connects the mathematics directly to the visualization in three ways:

Deep Component Cores (High Certainty):

In the regions colored with high intensity (where the contour values approach $1.0$), the expectation vector is highly polarized—for example, $\mathbb{E}[Z_i \mid X_i = x_i] \approx [0.98, 0.01, 0.01]^T$. This means there is near-absolute certainty that the point belongs to a specific cluster. Geometrically, these represent the high-density centers ($\mu_k$) of the individual Gaussian components.

Inter-Cluster Valleys (Ambiguity and Transition):

In the boundary zones where the colors transition into deep dark colors (valleys where the contour value drops toward $\frac{1}{K} \approx 0.33$), the expectation vector is spread out—for example, $\mathbb{E}[Z_i \mid X_i = x_i] \approx [0.48, 0.48, 0.04]^T$. This mathematically signals that the point sits on a "ridge" where two clusters are equally likely.

Continuous Probability Surface vs. Hard Splits:

Rather than showing static, hard-drawn boundary lines, the continuous gradient of the background contour map dynamically maps the rate of decay in belief as you move through the feature space. It serves as a visual proof that GMMs treat cluster membership not as a binary state, but as a continuous, mathematically rigorous expectation over a latent probability space.